In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()

SRC_DATASET = PROJECT_ROOT / "dataset"
DST_DATASET = PROJECT_ROOT / "dataset_9"

print("Project root:", PROJECT_ROOT)

print("\nItems in project folder:")
for p in PROJECT_ROOT.iterdir():
    print("-", p.name)

print("\nDataset check:")
print("dataset exists:", SRC_DATASET.exists())
print("train exists:", (SRC_DATASET / "train").exists())
print("val exists:", (SRC_DATASET / "val").exists())
print("test exists:", (SRC_DATASET / "test").exists())

Project root: d:\YOLO2

Items in project folder:
- chromedriver.exe
- dataset
- dataset_9
- images
- models
- requirements.txt
- runs
- scripts
- venv
- website.html
- YOLO.ipynb
- yolov8n-cls.pt

Dataset check:
dataset exists: True
train exists: True
val exists: True
test exists: False


In [ ]:
train_classes = sorted([p.name for p in (SRC_DATASET / "train").iterdir() if p.is_dir()])
val_classes = sorted([p.name for p in (SRC_DATASET / "val").iterdir() if p.is_dir()])

print("Train classes:", len(train_classes))
print(train_classes)

print("\nVal classes:", len(val_classes))
print(val_classes)

{0: 'Bicycle', 1: 'Bridge', 2: 'Bus', 3: 'Car', 4: 'Chimney', 5: 'Crosswalk', 6: 'Hydrant', 7: 'Motorcycle', 8: 'Other', 9: 'Palm', 10: 'Stair', 11: 'Traffic Light'}


In [2]:
from pathlib import Path
import shutil
import pandas as pd

TARGETS = {
    "Bicycle": ["bicycle", "bike", "Bicycle"],
    "Bridge": ["bridge", "Bridge"],
    "Bus": ["bus", "Bus"],
    "Car": ["car", "Car"],
    "Cross": ["cross", "crosswalk", "cross_walk", "Cross"],
    "Hydrant": ["hydrant", "fire_hydrant", "firehydrant", "Hydrant"],
    "Motorcycle": ["motorcycle", "motorbike", "motor_cycle", "Motorcycle"],
    "Stair": ["stair", "stairs", "Stair"],
    "Tlight": ["tlight", "traffic_light", "trafficlight", "traffic light", "Tlight"],
}

def normalize(name: str) -> str:
    return name.strip().lower().replace("-", "_").replace(" ", "_")

def find_source_class(split_dir: Path, aliases):
    existing_dirs = [p for p in split_dir.iterdir() if p.is_dir()]
    existing_map = {normalize(p.name): p for p in existing_dirs}

    for alias in aliases:
        key = normalize(alias)
        if key in existing_map:
            return existing_map[key]

    return None

def copy_images(src_class_dir: Path, dst_class_dir: Path):
    dst_class_dir.mkdir(parents=True, exist_ok=True)

    copied = 0
    for img_path in src_class_dir.glob("*.*"):
        if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".webp", ".bmp"]:
            continue

        dst_name = f"{src_class_dir.name}_{img_path.name}"
        shutil.copy2(img_path, dst_class_dir / dst_name)
        copied += 1

    return copied

if not (SRC_DATASET / "train").exists():
    raise FileNotFoundError(f"train folder not found: {SRC_DATASET / 'train'}")

if not (SRC_DATASET / "val").exists():
    raise FileNotFoundError(f"val folder not found: {SRC_DATASET / 'val'}")

if DST_DATASET.exists():
    print(f"Removing old dataset_9: {DST_DATASET}")
    shutil.rmtree(DST_DATASET)

summary = []

for split in ["train", "val"]:
    src_split = SRC_DATASET / split

    print(f"\nProcessing split: {split}")

    for target_name, aliases in TARGETS.items():
        src_class_dir = find_source_class(src_split, aliases)

        if src_class_dir is None:
            print(f"[MISSING] {target_name}")
            summary.append([split, target_name, "MISSING", 0])
            continue

        dst_class_dir = DST_DATASET / split / target_name
        count = copy_images(src_class_dir, dst_class_dir)

        print(f"{src_class_dir.name} -> {target_name}: {count} images")
        summary.append([split, target_name, src_class_dir.name, count])

df_summary = pd.DataFrame(
    summary,
    columns=["split", "target_class", "source_class", "image_count"]
)

summary_path = PROJECT_ROOT / "dataset_9_creation_summary.csv"
df_summary.to_csv(summary_path, index=False)

print("\nDone.")
print("Created:", DST_DATASET)
print("Summary saved to:", summary_path)

df_summary


Processing split: train
Bicycle -> Bicycle: 624 images
Bridge -> Bridge: 426 images
Bus -> Bus: 967 images
Car -> Car: 2846 images
Crosswalk -> Cross: 992 images
Hydrant -> Hydrant: 762 images
Motorcycle -> Motorcycle: 65 images
Stair -> Stair: 169 images
Traffic Light -> Tlight: 633 images

Processing split: val
Bicycle -> Bicycle: 156 images
Bridge -> Bridge: 107 images
Bus -> Bus: 242 images
Car -> Car: 712 images
Crosswalk -> Cross: 248 images
Hydrant -> Hydrant: 190 images
Motorcycle -> Motorcycle: 16 images
Stair -> Stair: 42 images
Traffic Light -> Tlight: 158 images

Done.
Created: d:\YOLO2\dataset_9
Summary saved to: d:\YOLO2\dataset_9_creation_summary.csv


,split,target_class,source_class,image_count
0,train,Bicycle,Bicycle,624
1,train,Bridge,Bridge,426
2,train,Bus,Bus,967
3,train,Car,Car,2846
4,train,Cross,Crosswalk,992
5,train,Hydrant,Hydrant,762
6,train,Motorcycle,Motorcycle,65
7,train,Stair,Stair,169
8,train,Tlight,Traffic Light,633
9,val,Bicycle,Bicycle,156


In [3]:
for split in ["train", "val"]:
    split_dir = DST_DATASET / split
    classes = sorted([p.name for p in split_dir.iterdir() if p.is_dir()])

    print("\n", split)
    print("Number of classes:", len(classes))
    print(classes)

    for c in classes:
        count = len(list((split_dir / c).glob("*.*")))
        print(f"{c}: {count}")


 train
Number of classes: 9
['Bicycle', 'Bridge', 'Bus', 'Car', 'Cross', 'Hydrant', 'Motorcycle', 'Stair', 'Tlight']
Bicycle: 624
Bridge: 426
Bus: 967
Car: 2846
Cross: 992
Hydrant: 762
Motorcycle: 65
Stair: 169
Tlight: 633

 val
Number of classes: 9
['Bicycle', 'Bridge', 'Bus', 'Car', 'Cross', 'Hydrant', 'Motorcycle', 'Stair', 'Tlight']
Bicycle: 156
Bridge: 107
Bus: 242
Car: 712
Cross: 248
Hydrant: 190
Motorcycle: 16
Stair: 42
Tlight: 158


In [4]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")

results = model.train(
    data=str(DST_DATASET),
    epochs=10,
    imgsz=224,
    batch=16,
    patience=4,
    project=str(PROJECT_ROOT / "runs" / "classify"),
    name="captcha_9cls_epoch10_imgsz224"
)

metrics = model.val()
print(metrics)

Ultralytics 8.4.80  Python-3.12.5 torch-2.12.1+cpu CPU (12th Gen Intel Core i7-12700H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\YOLO2\dataset_9, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=captcha_9cls_epoch10_imgsz224, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap

In [5]:
from ultralytics import YOLO

model = YOLO(r"models\best.pt")
print(model.names)

{0: 'Bicycle', 1: 'Bridge', 2: 'Bus', 3: 'Car', 4: 'Cross', 5: 'Hydrant', 6: 'Motorcycle', 7: 'Stair', 8: 'Tlight'}


In [7]:
## Dataset Analysis Outputs


In [8]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import random
import shutil

PROJECT_ROOT = Path.cwd()

DATASET_DIR = PROJECT_ROOT / "dataset_9"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
ANALYSIS_DIR = OUTPUTS_DIR / "analysis"
EVAL_DIR = OUTPUTS_DIR / "evaluation"

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_DIR)
print("Analysis output:", ANALYSIS_DIR)
print("Evaluation output:", EVAL_DIR)

Project root: d:\YOLO2
Dataset: d:\YOLO2\dataset_9
Analysis output: d:\YOLO2\outputs\analysis
Evaluation output: d:\YOLO2\outputs\evaluation


In [9]:
rows = []

for split in ["train", "val"]:
    split_dir = DATASET_DIR / split

    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue

        images = [
            p for p in class_dir.glob("*.*")
            if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
        ]

        rows.append({
            "split": split,
            "class": class_dir.name,
            "count": len(images)
        })

df_counts = pd.DataFrame(rows)
df_counts.to_csv(ANALYSIS_DIR / "dataset_counts.csv", index=False)

df_counts

,split,class,count
0,train,Bicycle,624
1,train,Bridge,426
2,train,Bus,967
3,train,Car,2846
4,train,Cross,992
5,train,Hydrant,762
6,train,Motorcycle,65
7,train,Stair,169
8,train,Tlight,633
9,val,Bicycle,156


In [10]:
pivot_counts = df_counts.pivot(index="class", columns="split", values="count").fillna(0)

ax = pivot_counts.plot(kind="bar", figsize=(12, 6))
ax.set_title("Dataset Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Number of Images")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.savefig(ANALYSIS_DIR / "dataset_counts.png", dpi=200)
plt.show()

<Figure size 1200x600 with 1 Axes>

In [11]:
classes = sorted([p.name for p in (DATASET_DIR / "train").iterdir() if p.is_dir()])

samples_per_class = 3
thumb_size = (160, 160)

fig, axes = plt.subplots(
    len(classes),
    samples_per_class,
    figsize=(samples_per_class * 3, len(classes) * 2.6)
)

for row_idx, cls in enumerate(classes):
    class_dir = DATASET_DIR / "train" / cls
    images = [
        p for p in class_dir.glob("*.*")
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ]

    chosen = random.sample(images, min(samples_per_class, len(images)))

    for col_idx in range(samples_per_class):
        ax = axes[row_idx][col_idx]

        if col_idx < len(chosen):
            img = Image.open(chosen[col_idx]).convert("RGB")
            img = ImageOps.fit(img, thumb_size)
            ax.imshow(img)

        ax.axis("off")

        if col_idx == 0:
            ax.set_title(cls, fontsize=11)

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "sample_grid.png", dpi=200)
plt.show()

<Figure size 900x2340 with 27 Axes>

In [12]:
website_targets = [
    "Bicycle",
    "Bridge",
    "Bus",
    "Car",
    "Cross",
    "Hydrant",
    "Motorcycle",
    "Stair",
    "Tlight",
]

with open(ANALYSIS_DIR / "website_targets.txt", "w", encoding="utf-8") as f:
    for t in website_targets:
        f.write(t + "\n")

print("Saved:", ANALYSIS_DIR / "website_targets.txt")

Saved: d:\YOLO2\outputs\analysis\website_targets.txt


In [13]:
src_summary = PROJECT_ROOT / "dataset_9_creation_summary.csv"
dst_summary = ANALYSIS_DIR / "dataset_9_creation_summary.csv"

if src_summary.exists():
    shutil.copy2(src_summary, dst_summary)
    print("Copied:", dst_summary)
else:
    print("dataset_9_creation_summary.csv was not found.")

Copied: d:\YOLO2\outputs\analysis\dataset_9_creation_summary.csv


In [14]:
## Model Evaluation Outputs

In [16]:
from ultralytics import YOLO

MODEL_PATH = PROJECT_ROOT / "models" / "best.pt"

model = YOLO(str(MODEL_PATH))
print(model.names)

{0: 'Bicycle', 1: 'Bridge', 2: 'Bus', 3: 'Car', 4: 'Cross', 5: 'Hydrant', 6: 'Motorcycle', 7: 'Stair', 8: 'Tlight'}


In [17]:
val_dir = DATASET_DIR / "val"

pred_rows = []

for class_dir in sorted(val_dir.iterdir()):
    if not class_dir.is_dir():
        continue

    true_label = class_dir.name

    image_paths = [
        p for p in class_dir.glob("*.*")
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ]

    for img_path in image_paths:
        result = model.predict(
            source=str(img_path),
            imgsz=224,
            verbose=False
        )[0]

        top1_idx = int(result.probs.top1)
        pred_label = result.names[top1_idx]
        confidence = float(result.probs.top1conf)

        pred_rows.append({
            "image_path": str(img_path.relative_to(PROJECT_ROOT)),
            "true_label": true_label,
            "pred_label": pred_label,
            "confidence": confidence,
            "correct": true_label == pred_label
        })

df_preds = pd.DataFrame(pred_rows)
df_preds.to_csv(EVAL_DIR / "predictions.csv", index=False)

print("Total validation images:", len(df_preds))
print("Accuracy:", df_preds["correct"].mean())

df_preds.head()

Total validation images: 1871
Accuracy: 0.9139497594869054


,image_path,true_label,pred_label,confidence,correct
0,dataset_9\val\Bicycle\Bicycle_Bicycle (661).png,Bicycle,Bicycle,0.993068,True
1,dataset_9\val\Bicycle\Bicycle_Bicycle (662).png,Bicycle,Bicycle,0.917705,True
2,dataset_9\val\Bicycle\Bicycle_Bicycle (663).png,Bicycle,Bicycle,0.999588,True
3,dataset_9\val\Bicycle\Bicycle_Bicycle (664).png,Bicycle,Bicycle,0.999208,True
4,dataset_9\val\Bicycle\Bicycle_Bicycle (665).png,Bicycle,Bicycle,0.999924,True


In [18]:
from sklearn.metrics import classification_report

labels = sorted(df_preds["true_label"].unique())

report = classification_report(
    df_preds["true_label"],
    df_preds["pred_label"],
    labels=labels,
    output_dict=True,
    zero_division=0
)

df_report = pd.DataFrame(report).transpose()
df_report.to_csv(EVAL_DIR / "classification_report.csv")

df_report

,precision,recall,f1-score,support
Bicycle,0.924051,0.935897,0.929936,156.00000
Bridge,0.894737,0.794393,0.841584,107.00000
Bus,0.909465,0.913223,0.911340,242.00000
Car,0.910490,0.914326,0.912404,712.00000
Cross,0.890625,0.919355,0.904762,248.00000
Hydrant,1.000000,1.000000,1.000000,190.00000
Motorcycle,0.800000,0.750000,0.774194,16.00000
Stair,0.826087,0.904762,0.863636,42.00000
Tlight,0.908497,0.879747,0.893891,158.00000
accuracy,0.913950,0.913950,0.913950,0.91395


In [19]:
from sklearn.metrics import confusion_matrix

labels = sorted(df_preds["true_label"].unique())

cm = confusion_matrix(
    df_preds["true_label"],
    df_preds["pred_label"],
    labels=labels
)

df_cm = pd.DataFrame(cm, index=labels, columns=labels)
df_cm.to_csv(EVAL_DIR / "confusion_matrix.csv")

df_cm

,Bicycle,Bridge,Bus,Car,Cross,Hydrant,Motorcycle,Stair,Tlight
Bicycle,146,0,0,4,3,0,3,0,0
Bridge,0,85,1,17,1,0,0,2,1
Bus,2,1,221,16,1,0,0,0,1
Car,5,7,18,651,20,0,0,4,7
Cross,3,2,1,9,228,0,0,1,4
Hydrant,0,0,0,0,0,190,0,0,0
Motorcycle,2,0,0,1,1,0,12,0,0
Stair,0,0,2,1,0,0,0,38,1
Tlight,0,0,0,16,2,0,0,1,139


In [20]:
fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(cm)

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)

ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("Confusion Matrix")

for i in range(len(labels)):
    for j in range(len(labels)):
        value = cm[i, j]
        ax.text(j, i, str(value), ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=ax)
plt.tight_layout()

plt.savefig(EVAL_DIR / "confusion_matrix.png", dpi=200)
plt.show()

<Figure size 1000x800 with 2 Axes>

In [21]:
error_rows = []

for cls in labels:
    cls_df = df_preds[df_preds["true_label"] == cls]
    total = len(cls_df)
    wrong = int((cls_df["correct"] == False).sum())
    correct = int((cls_df["correct"] == True).sum())
    error_rate = wrong / total if total > 0 else 0

    error_rows.append({
        "class": cls,
        "total": total,
        "correct": correct,
        "wrong": wrong,
        "error_rate": error_rate
    })

df_errors = pd.DataFrame(error_rows).sort_values("error_rate", ascending=False)
df_errors.to_csv(EVAL_DIR / "per_class_errors.csv", index=False)

df_errors

,class,total,correct,wrong,error_rate
6,Motorcycle,16,12,4,0.250000
1,Bridge,107,85,22,0.205607
8,Tlight,158,139,19,0.120253
7,Stair,42,38,4,0.095238
2,Bus,242,221,21,0.086777
3,Car,712,651,61,0.085674
4,Cross,248,228,20,0.080645
0,Bicycle,156,146,10,0.064103
5,Hydrant,190,190,0,0.000000


In [22]:
df_wrong = df_preds[df_preds["correct"] == False].copy()
df_wrong.to_csv(EVAL_DIR / "wrong_predictions.csv", index=False)

print("Wrong predictions:", len(df_wrong))
df_wrong.head(20)

Wrong predictions: 161


,image_path,true_label,pred_label,confidence,correct
24,dataset_9\val\Bicycle\Bicycle_Bicycle (683).png,Bicycle,Car,0.427307,False
36,dataset_9\val\Bicycle\Bicycle_Bicycle (694).png,Bicycle,Cross,0.916740,False
44,dataset_9\val\Bicycle\Bicycle_Bicycle (700).png,Bicycle,Cross,0.783178,False
48,dataset_9\val\Bicycle\Bicycle_Bicycle (704).png,Bicycle,Car,0.406062,False
51,dataset_9\val\Bicycle\Bicycle_Bicycle (707).png,Bicycle,Cross,0.786428,False
63,dataset_9\val\Bicycle\Bicycle_Bicycle (718).png,Bicycle,Motorcycle,0.826652,False
74,dataset_9\val\Bicycle\Bicycle_Bicycle (728).png,Bicycle,Car,0.838201,False
98,dataset_9\val\Bicycle\Bicycle_Bicycle (75).png,Bicycle,Motorcycle,0.749975,False
119,dataset_9\val\Bicycle\Bicycle_Bicycle (769).png,Bicycle,Car,0.381405,False
144,dataset_9\val\Bicycle\Bicycle_Bicycle (89).png,Bicycle,Motorcycle,0.964698,False
